# Machine Learning Model to Predict Late Delivery
This notebook flattens data from `shop.db` into a single analytical view and trains a Random Forest classifier.


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. Connect to the Operational Database
conn = sqlite3.connect('shop.db')


## 1. Data Extraction & "Warehouse" Creation
We will extract records from `orders`, join them with `customers`, `shipments`, and aggregated `order_items`.
- `orders` forms the base (for orders that have shipments, fulfilled=1).
- `customers` provides demographic info.
- `shipments` provides our target label `late_delivery`.
- `order_items` provides total quantity per order.


In [ ]:
warehouse_query = '''
    SELECT 
        o.order_id,
        o.customer_id,
        o.order_subtotal,
        o.shipping_fee,
        o.tax_amount,
        o.order_total,
        o.risk_score,
        o.device_type,
        o.payment_method,
        c.customer_segment,
        c.loyalty_tier,
        s.late_delivery,
        COALESCE(items.total_items, 0) AS total_items
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN shipments s ON o.order_id = s.order_id
    LEFT JOIN (
        SELECT order_id, SUM(quantity) as total_items
        FROM order_items
        GROUP BY order_id
    ) items ON o.order_id = items.order_id
    WHERE o.fulfilled = 1
'''
df = pd.read_sql_query(warehouse_query, conn)
display(df.head())


## 2. Feature Engineering
We will process categorical fields into dummy variables.


In [ ]:
# Drop identifiers that shouldn't be used for predicting
features = df.drop(columns=['order_id', 'customer_id', 'late_delivery'])
target = df['late_delivery']

# One-hot encode categorical features
categorical_cols = ['device_type', 'payment_method', 'customer_segment', 'loyalty_tier']
features_encoded = pd.get_dummies(features, columns=categorical_cols, drop_first=True)

# Prepare Train/Test sets
X_train, X_test, y_train, y_test = train_test_split(features_encoded, target, test_size=0.2, random_state=42)

print("Training Data Shape:", X_train.shape)


## 3. Model Training & Evaluation


In [ ]:
# Train Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
preds = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))


## 4. Artifact Preservation
Export the trained model and the feature names so inference logic matches perfectly.


In [ ]:
model_artifacts = {
    'model': clf,
    'features': list(X_train.columns)
}
joblib.dump(model_artifacts, 'model.joblib')
print("Model and features exported to model.joblib successfully!")


## 5. Establishing the Database Contract
We must ensure the `order_predictions` table in `shop.db` matches exactly what the web app deployment team expects.
We will run `PRAGMA table_info` to validate the schema.


In [ ]:
cursor = conn.cursor()
cursor.execute("PRAGMA table_info(order_predictions);")
schema = cursor.fetchall()
print("Schema for order_predictions table:")
for col in schema:
    print(f"Column ID: {col[0]}, Name: {col[1]}, Type: {col[2]}")
    
conn.close()
